### 1. Import Libraries

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns
import time 

sns.set(style="whitegrid")

### 2. Load Datasets

Extract the `GHGs_by_Sector_and_State_2012-2022.xlsx` (U.S. greenhouse gas emissions per sector and state from 2012-2022) main dataset:

In [2]:
current_directory = os.getcwd()
ghg_filename = "GHGs_by_Sector_and_State_2012-2022.xlsx"

ghg_main_df = pd.read_excel(
    os.path.join(current_directory, "datasets", ghg_filename),
    sheet_name = 'Main'
)

ghg_main_df.head()

,Flowable,Sector,State,Year,FlowAmount
0,Carbon dioxide,111,AK,2012,8.349448e+07
1,Carbon dioxide,111,AK,2013,1.259086e+08
2,Carbon dioxide,111,AK,2014,1.030956e+08
3,Carbon dioxide,111,AK,2015,9.149795e+07
4,Carbon dioxide,111,AK,2016,8.142637e+07


Extract the `GHGs_by_Sector_and_State_2012-2022.xlsx` (U.S. greenhouse gas emissions per sector and state from 2012-2022) `Sector` table:

In [3]:
ghg_sectors_df = pd.read_excel(
    os.path.join(current_directory, "datasets", ghg_filename),
    sheet_name = 'Sectors'
)

ghg_sectors_df.head()

,Sector,SectorName
0,111,Crop Production
1,112,Animal Production and Aquaculture
2,113,Forestry and Logging
3,114,"Fishing, Hunting and Trapping"
4,115,Support Activities for Agriculture and Forestry


Now that `Sectors` data is extracted, both datasets can be combined based on the `Sector` column which is the primary key of the `Sectors` sheet and foreign key of the `Main` sheet:

In [4]:
ghg_df = ghg_main_df.merge(
    ghg_sectors_df, 
    on='Sector',
    how='left'
)

ghg_df.head()

,Flowable,Sector,State,Year,FlowAmount,SectorName
0,Carbon dioxide,111,AK,2012,8.349448e+07,Crop Production
1,Carbon dioxide,111,AK,2013,1.259086e+08,Crop Production
2,Carbon dioxide,111,AK,2014,1.030956e+08,Crop Production
3,Carbon dioxide,111,AK,2015,9.149795e+07,Crop Production
4,Carbon dioxide,111,AK,2016,8.142637e+07,Crop Production


From above, all variables are loaded properly with correct headers. Now, values in each column will be checked on whether they are non-null values and that they align with the expected header variable type:
- Flowable (nominal categorical): type of greenhouse gas
- Sector (nominal categorical, foreign key to sector table): NAICs industry code (though each code is represented by an integer value, each code represents a category)
- State (nominal categorical): two-letter U.S. state abbreviation
- Year (interval numerical): year 
- FlowAmount (ratio numerical): greenhouse gas emissions in kg 

In [5]:
ghg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503962 entries, 0 to 503961
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Flowable    503962 non-null  object 
 1   Sector      503962 non-null  object 
 2   State       503962 non-null  object 
 3   Year        503962 non-null  int64  
 4   FlowAmount  503962 non-null  float64
 5   SectorName  503962 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 23.1+ MB


From above, all values in all columns have the proper variable type and are non-null values. The `Greenhouse Gases` dataset has been loaded properly into a pandas DataFrame with appropriate headers, and the data types are consistent with expectations for each variable. 

### 3. Engineering a new feature: GWPAmount

Global Warming Potential (GWP) is how the Intergovernmental Panel on Climate Change (IPCC) compare the global warming impact of different greenhouse gases - the GWP measures how much energy 1 emitted ton of the gas will absorb given a period of time (typically 100 years) relative to 1 emitted ton of $CO_2$ (source: https://www.epa.gov/ghgemissions/understanding-global-warming-potentials). Therefore larger GWP values correspond to higher effects on the Earth's warming. 

`co2e.excel` was transcribed from the IPCC Greenhouse Gas Protocol to obtain the GWP values for the greenhouse gases in this dataset (source: https://ghgprotocol.org/sites/default/files/2024-08/Global-Warming-Potential-Values%20%28August%202024%29.pdf:

In [6]:
co2e_df = pd.read_excel(os.path.join(os.getcwd(), "datasets", "co2e.xlsx"))

In [7]:
co2e_df

,Flowable,IPCC_Name,GWP
0,Carbon dioxide,Carbon dioxide,1.000000
1,Carbon tetrafluoride,PFC-14,7380.000000
2,HFC-125,HFC-125,3740.000000
3,HFC-134a,HFC-134a,1530.000000
4,HFC-143a,HFC-143a,5810.000000
5,HFC-23,HFC-23,14600.000000
6,HFC-236fa,HFC-236fa,8690.000000
7,HFC-32,HFC-32,771.000000
8,"HFCs and PFCs, unspecified",NaN,2956.239675
9,Hexafluoroethane,PFC-116,12400.000000


*Please note that for "HFCs and PFCs, unspecified" the value 2956.239675 is calculated as $GWP_{unspecified}$:

$GWP_{unspecified} = \frac{1}{n}\sum_{i=1}^{n}GWP_i$

Using the `co2e_df` table, emissions records will be matched with their corresponding GWPs: 

In [8]:
ghg_df = ghg_df.merge(
    co2e_df[['Flowable', 'GWP']], 
    on='Flowable',
    how='left'
)

ghg_df.head()

,Flowable,Sector,State,Year,FlowAmount,SectorName,GWP
0,Carbon dioxide,111,AK,2012,8.349448e+07,Crop Production,1.0
1,Carbon dioxide,111,AK,2013,1.259086e+08,Crop Production,1.0
2,Carbon dioxide,111,AK,2014,1.030956e+08,Crop Production,1.0
3,Carbon dioxide,111,AK,2015,9.149795e+07,Crop Production,1.0
4,Carbon dioxide,111,AK,2016,8.142637e+07,Crop Production,1.0


A new column `GWPAmount` will be calculated for each emissions record, representing the $CO_2$ equivalent mass of gas emitted: $GWPAmount = GWP \times FlowAmount$

In [13]:
ghg_df['GWPAmount'] = ghg_df['GWP'] * ghg_df['FlowAmount']

ghg_df.head()

,Flowable,Sector,State,Year,FlowAmount,SectorName,GWP,GWPAmount
0,Carbon dioxide,111,AK,2012,8.349448e+07,Crop Production,1.0,8.349448e+07
1,Carbon dioxide,111,AK,2013,1.259086e+08,Crop Production,1.0,1.259086e+08
2,Carbon dioxide,111,AK,2014,1.030956e+08,Crop Production,1.0,1.030956e+08
3,Carbon dioxide,111,AK,2015,9.149795e+07,Crop Production,1.0,9.149795e+07
4,Carbon dioxide,111,AK,2016,8.142637e+07,Crop Production,1.0,8.142637e+07


As seen above, `GWPAmount` has been successfully calculated for each record of the dataset.

### 4. Numerical PreProcessing - Restructuring the Dataset into a Sector, State, and Time-Specific Wide Format

To make the dataset more suitable for modelling and analysis, features need to be in a fixed, tabular format. The dataset can be pivotted from a long format (one row per emission record with `FlowAmount` and `GWPAmount` as two numerical features) into a wide format (each row is represented by a unique combination of `SectorName`, `State`, and `Year` with GHG-specific `FlowAmount` and `GWPAmount` values as multiple different features per record):

In [21]:
ghg_reshaped_df = ghg_df.pivot_table(
    index=['SectorName', 'State', 'Year'],
    columns='Flowable',
    values=['FlowAmount', 'GWPAmount'],
    aggfunc='sum'
)

ghg_reshaped_df.columns = [f"{metric}_{flowable}" for metric, flowable in ghg_reshaped_df.columns]

ghg_reshaped_df = ghg_reshaped_df.reset_index()


In [24]:
ghg_df.head()

,Flowable,Sector,State,Year,FlowAmount,SectorName,GWP,GWPAmount
0,Carbon dioxide,111,AK,2012,8.349448e+07,Crop Production,1.0,8.349448e+07
1,Carbon dioxide,111,AK,2013,1.259086e+08,Crop Production,1.0,1.259086e+08
2,Carbon dioxide,111,AK,2014,1.030956e+08,Crop Production,1.0,1.030956e+08
3,Carbon dioxide,111,AK,2015,9.149795e+07,Crop Production,1.0,9.149795e+07
4,Carbon dioxide,111,AK,2016,8.142637e+07,Crop Production,1.0,8.142637e+07


In [26]:
ghg_reshaped_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66187 entries, 0 to 66186
Data columns (total 39 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SectorName                             66187 non-null  object 
 1   State                                  66187 non-null  object 
 2   Year                                   66187 non-null  int64  
 3   FlowAmount_Carbon dioxide              66165 non-null  float64
 4   FlowAmount_Carbon tetrafluoride        880 non-null    float64
 5   FlowAmount_HFC-125                     52173 non-null  float64
 6   FlowAmount_HFC-134a                    52173 non-null  float64
 7   FlowAmount_HFC-143a                    52173 non-null  float64
 8   FlowAmount_HFC-23                      386 non-null    float64
 9   FlowAmount_HFC-236fa                   52173 non-null  float64
 10  FlowAmount_HFC-32                      52173 non-null  float64
 11  Fl

From above, the dataset has been successfully been converted from long format with few features into wide format (grouped by `SectorName`, `State`, and `Year`) with numerous GHG-specific features. From some GHG-specific columns (for example `GWPAmount_Perfluorobutane`), there are many null (missing) values) which indicates sparse data for these GHGs across sectors, states, and years. 

Nonetheless, this reshaping will lay the foundation for further preprocessing, feature engineering and scaling. 

### Address missing values, outliers, and inconsistencies.

### Create target variable ('Total emissions'), make flowable features per record (group flowables by state and year, add each GHG/flowable type as a different column)

### Z-Score Scale numerical values